# Mettre en place son environnement de développement Python

> **Objectif** : comprendre comment installer Python, gérer ses dépendances et travailler dans un environnement reproductible — en utilisant les outils modernes de 2026.



## 1. Petit historique de la gestion des paquets Python

Python existe depuis 1991, et la façon d'installer des bibliothèques a beaucoup évolué. Voici les grandes étapes :

| Époque | Outil | Ce qu'il faisait |
|--------|-------|-----------------|
| ~2004 | **`easy_install`** (setuptools) | Premier installateur automatique de paquets Python. On tapait `easy_install numpy` et ça allait chercher le paquet sur Internet. Pas de désinstallation propre ! |
| ~2008 | **`pip`** | Remplaçant d'`easy_install` : plus fiable, permet de désinstaller, utilise `requirements.txt` pour lister les dépendances. C'est devenu **le** standard. |
| ~2012 | **`conda`** (Anaconda) | Gestionnaire de paquets + environnements, très populaire en data science. Installe aussi des bibliothèques non-Python (C, Fortran…). Lourd (~3 Go pour Anaconda). |
| ~2018 | **`pipenv`**, **`poetry`** | Outils "tout-en-un" : environnement virtuel + gestion des dépendances + fichier de lock. Poetry a introduit `pyproject.toml` comme fichier de configuration unique. |
| ~2024 | **`uv`** | Outil ultra-rapide écrit en Rust. Remplace `pip`, `venv`, `pyenv`, `pipx`… en un seul outil. C'est ce qu'on va utiliser dans ce cours. |

## 2. Pourquoi des environnements virtuels ?

Imaginez que vous travaillez sur deux projets :
- Le **Projet A** a besoin de `pandas 1.5`
- Le **Projet B** a besoin de `pandas 2.2`

Si vous installez tout "en vrac" sur votre machine, l'un des deux projets va casser.

**Un environnement virtuel** est un dossier isolé qui contient sa propre copie de Python et de ses bibliothèques. Chaque projet a le sien, et ils ne se marchent pas dessus.

```
Machine
├── Projet A/
│   └── .venv/          ← pandas 1.5, numpy 1.24
├── Projet B/
│   └── .venv/          ← pandas 2.2, numpy 2.0
```

**Règle d'or** : un projet = un environnement virtuel. Toujours.

## 3. Le fichier `pyproject.toml` — la carte d'identité de votre projet

Depuis 2024, le standard Python pour décrire un projet est le fichier **`pyproject.toml`** (défini par les PEP 517, 518 et 621).

C'est un fichier texte qui contient :
- Le **nom** du projet
- La **version de Python** requise
- La **liste des dépendances** (les bibliothèques dont votre code a besoin)

Exemple minimal :

```toml
[project]
name = "mon-projet-data"
version = "0.1.0"
requires-python = ">=3.12"
dependencies = [
    "pandas>=2.2",
    "matplotlib>=3.9",
    "jupyterlab>=4.0",
]
```

> **Avant**, on utilisait `requirements.txt` (une simple liste de paquets). `pyproject.toml` le remplace en étant plus structuré et standardisé.
> Un fichier `requirements.txt` reste utile pour déployer sur certains serveurs, mais pour un projet local `pyproject.toml` est le standard.

## 4. `uv` — l'outil moderne pour tout gérer

**[uv](https://docs.astral.sh/uv/)** est un outil créé par Astral (les mêmes qui font `ruff`, le linter/formateur Python). Écrit en Rust, il est **extrêmement rapide** et remplace à lui seul plusieurs outils :

| Besoin | Avant (plusieurs outils) | Maintenant (uv) |
|--------|--------------------------|-----------------|
| Installer Python | `pyenv`, téléchargement manuel | `uv python install 3.12` |
| Créer un environnement virtuel | `python -m venv .venv` | `uv venv` (ou automatique) |
| Installer des paquets | `pip install pandas` | `uv add pandas` |
| Lancer un script | `python script.py` | `uv run script.py` |
| Reproduire un environnement | `pip install -r requirements.txt` | `uv sync` |
| Lancer un outil CLI | `pipx install ruff` | `uv tool install ruff`/`uvx tool install ruff` |

### 4.1 Installer `uv`

Sur **macOS / Linux** :
```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

Sur **Windows** :
```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

Vérifier que ça marche :
```bash
uv --version
```

> [!TIP]
> **Vous utilisez déjà un gestionnaire de paquets système ?** Vous pouvez aussi installer `uv` par ce biais :
>
> | OS | Commande |
> |-----|----------|
> | **macOS** (Homebrew) | `brew install uv` |
> | **Windows** (Chocolatey) | `choco install uv` |
> | **Windows** (Scoop) | `scoop install uv` |
>
> Sur les autres distributions Linux, le script `curl` ci-dessus reste la méthode la plus simple.

### 4.2 Créer un nouveau projet

```bash
uv init mon-projet-data
cd mon-projet-data
```

Cette commande crée :
- Un fichier `pyproject.toml` pré-rempli
- Un fichier `hello.py` d'exemple
- Un dossier `.venv/` (l'environnement virtuel, créé automatiquement)
- Un fichier `uv.lock` (les versions exactes des dépendances, pour la reproductibilité)

### 4.3 Ajouter des dépendances

```bash
uv add pandas matplotlib jupyterlab
```

Cette commande :
1. Ajoute `pandas`, `matplotlib` et `jupyterlab` dans le `pyproject.toml`
2. Les installe dans le `.venv/`
3. Met à jour le `uv.lock` avec les versions exactes

Pour ajouter une dépendance avec une contrainte de version :
```bash
uv add "pandas>=2.2"
```

Pour retirer une dépendance :
```bash
uv remove matplotlib
```

### 4.4 Lancer des commandes dans l'environnement

`uv run` exécute une commande en utilisant automatiquement l'environnement virtuel du projet. Pas besoin d'activer manuellement le `.venv`.

```bash
# Lancer un script Python
uv run python mon_script.py

# Lancer Jupyter Lab
uv run jupyter lab

# Lancer un module
uv run python -m pytest
```

> **Note** : si vous préférez activer le `.venv` à l'ancienne, c'est toujours possible :
> ```bash
> source .venv/bin/activate    # macOS / Linux
> .venv\Scripts\activate       # Windows
> ```

### 4.5 Reproduire un environnement (partager son projet)

Quand un collègue récupère votre projet, il lui suffit de lancer :

```bash
uv sync
```

Cette commande lit le `uv.lock` et installe **exactement** les mêmes versions de toutes les dépendances. C'est la garantie que le code marchera de la même façon sur sa machine.

**En résumé, le workflow quotidien avec `uv` :**

```
uv init mon-projet       # 1. Créer le projet (une seule fois)
cd mon-projet
uv add pandas jupyterlab # 2. Ajouter les dépendances
uv run jupyter lab       # 3. Travailler !
```

Quand un collègue rejoint le projet :
```
git clone <url-du-projet>
cd mon-projet
uv sync                  # Tout est installé, prêt à travailler
```

### 4.6 Installer une version de Python avec `uv`

Pas besoin d'aller sur python.org ou d'utiliser `pyenv`. `uv` peut installer Python directement :

```bash
# Installer Python 3.12
uv python install 3.12

# Voir les versions disponibles
uv python list

# Créer un projet avec une version spécifique
uv init mon-projet --python 3.12
```

### 4.7 Exporter vers `requirements.txt`

Certains environnements (serveurs, Binder, Google Colab…) ne supportent pas `uv` et ont besoin d'un `requirements.txt` classique. On peut l'exporter :

```bash
uv export --format requirements-txt > requirements.txt
```

Cela génère un fichier `requirements.txt` avec les versions exactes, à partir du `uv.lock`.

### 4.8 Récapitulatif des commandes `uv`

| Commande | Description |
|----------|-------------|
| `uv init <nom>` | Crée un nouveau projet avec `pyproject.toml` |
| `uv add <paquet>` | Ajoute une dépendance |
| `uv remove <paquet>` | Retire une dépendance |
| `uv sync` | Installe toutes les dépendances du `uv.lock` |
| `uv run <commande>` | Lance une commande dans l'environnement du projet |
| `uv python install <version>` | Installe une version de Python |
| `uv python list` | Liste les versions de Python disponibles |
| `uv lock` | Met à jour le fichier `uv.lock` |
| `uv export` | Exporte les dépendances (ex. vers `requirements.txt`) |
| `uv tool install <outil>` | Installe un outil CLI (comme `pipx`) |

***

## Pour aller plus loin : l'histoire complète du packaging Python

<details>
<summary><strong>Cliquez pour déplier cette section</strong></summary>

### L'ère du Far West (~2000–2008)

À ses débuts, Python n'avait **aucun** système officiel pour installer des bibliothèques tierces. On téléchargeait un `.tar.gz`, on le décompressait, et on lançait `python setup.py install`. Chaque bibliothèque venait avec un fichier `setup.py` qui contenait les instructions d'installation.

**`distutils`** (livré avec Python) fournissait les briques de base pour `setup.py`, mais sans gestion des dépendances ni de dépôt central.

**`setuptools`** (2004) a amélioré `distutils` en ajoutant :
- `easy_install` : une commande pour télécharger et installer des paquets depuis PyPI (Python Package Index, le dépôt central)
- Le format `.egg` : un ancêtre du `.whl` (wheel) pour distribuer des paquets pré-compilés
- La gestion des dépendances dans `setup.py`

Problème : `easy_install` ne savait pas désinstaller proprement, ne vérifiait pas les conflits de versions, et le format `.egg` était peu standardisé.

### L'arrivée de `pip` (~2008) — PEP 453

**`pip`** ("Pip Installs Packages") a été créé pour corriger les défauts d'`easy_install` :
- Désinstallation propre (`pip uninstall`)
- Format `requirements.txt` pour lister les dépendances d'un projet
- Installation depuis PyPI, depuis Git, ou depuis un fichier local
- Support du format **wheel** (`.whl`) qui remplace le `.egg` — plus rapide, plus fiable

La **PEP 453** (2013) a intégré `pip` dans l'installation standard de Python (via `ensurepip`). Depuis Python 3.4, `pip` est installé automatiquement.

`pip` reste aujourd'hui l'installateur de référence, mais il a des limites :
- Pas de résolution de dépendances fiable avant la version 20.3 (2020)
- Pas de fichier de lock (le `requirements.txt` ne garantit pas la reproductibilité exacte)
- Pas de gestion d'environnement virtuel intégrée

### `requirements.txt` et `requirements.in`

Le fichier `requirements.txt` est le format historique pour déclarer les dépendances :

```
pandas==2.2.1
matplotlib>=3.9.0
numpy>=1.26,<2.0
```

Mais ce format a des limites :
- On mélange souvent les dépendances **directes** (ce que je veux) et **transitives** (ce que mes dépendances veulent)
- Pas de distinction entre dépendances de production et de développement

L'outil **`pip-tools`** a introduit la convention `requirements.in` / `requirements.txt` :
- `requirements.in` : vos dépendances directes (ce que vous écrivez)
- `requirements.txt` : le fichier généré automatiquement avec toutes les versions exactes (le "lock")

C'est un ancêtre conceptuel du couple `pyproject.toml` / `uv.lock`.

### Les environnements virtuels — `venv` et `virtualenv`

**`virtualenv`** (2007) a été le premier outil pour créer des environnements isolés. La **PEP 405** (2012) a intégré cette fonctionnalité directement dans Python sous le nom **`venv`** :

```bash
python -m venv .venv
source .venv/bin/activate
```

C'est simple et ça marche, mais c'est une opération manuelle séparée de l'installation des paquets. Les outils modernes (uv, poetry…) combinent les deux.

### `conda` et Anaconda (~2012)

**Conda** est un gestionnaire de paquets et d'environnements créé par Anaconda Inc., pensé pour la data science :
- Installe des paquets Python **et** non-Python (bibliothèques C, Fortran, R…)
- Gère ses propres environnements virtuels
- Distribué via **Anaconda** (~3 Go, avec +250 paquets pré-installés) ou **Miniconda** (version légère)

Pourquoi c'était populaire :
- `numpy`, `scipy`, `scikit-learn` nécessitaient des compilations C/Fortran complexes — conda fournissait des binaires pré-compilés
- Environnement tout-en-un pour les débutants

Pourquoi on s'en éloigne :
- Lourd et lent (résolution de dépendances parfois très longue)
- Écosystème parallèle à PyPI (les paquets conda ≠ les paquets pip, ce qui crée des conflits)
- Changement de licence d'Anaconda (payant pour les grandes entreprises depuis 2024)
- Les problèmes de compilation C sont résolus par les wheels : `pip install numpy` fonctionne parfaitement en 2026

> **Note** : `conda-forge` (communautaire) et `mamba` (remplacement rapide de conda) existent toujours et restent pertinents pour des cas scientifiques pointus (dépendances système lourdes).

### `pyenv` — gérer plusieurs versions de Python

**`pyenv`** (uniquement macOS/Linux) permet d'installer et basculer entre plusieurs versions de Python :

```bash
pyenv install 3.12.2
pyenv global 3.12.2
```

`uv python install` remplace cette fonctionnalité en 2026.

### Homebrew (macOS)

Sur macOS, on peut installer Python via **Homebrew** (`brew install python`). C'est pratique pour avoir un Python système, mais ce n'est pas un gestionnaire de paquets Python : on ne fait pas `brew install pandas`. Homebrew sert uniquement à installer l'interpréteur Python lui-même (et d'autres outils système).

### `pipenv` (~2017)

**`pipenv`** a été le premier outil "tout-en-un" populaire :
- Combine `pip` + `virtualenv` en une seule commande
- Introduit le `Pipfile` et le `Pipfile.lock` (versions exactes)

```bash
pipenv install pandas
pipenv shell
```

Il n'a pas été massivement adopté : développement lent, résolution de dépendances parfois très longue, et le `Pipfile` n'est pas devenu un standard officiel.

### `poetry` (~2018)

**Poetry** a apporté une vision plus moderne :
- Fichier unique `pyproject.toml` pour tout (métadonnées, dépendances, configuration des outils)
- Fichier de lock (`poetry.lock`) pour la reproductibilité exacte
- Résolution de dépendances plus fiable
- Publication de paquets sur PyPI intégrée

```bash
poetry init
poetry add pandas
poetry run python script.py
```

Poetry a largement contribué à populariser `pyproject.toml`, mais :
- Il est relativement lent (écrit en Python)
- Il impose certaines conventions sur la structure du projet
- Il n'installe pas Python (il faut un Python déjà présent)

### `hatch` et `PDM`

**Hatch** (maintenu par le responsable du packaging chez PyPA) et **PDM** sont d'autres outils "modernes" :

- **Hatch** : orienté publication de paquets, gestion d'environnements multiples, supporte les PEP standards. Utilisé en interne par de grands projets.
- **PDM** : un des premiers à supporter la **PEP 582** (dossier `__pypackages__` local, finalement rejetée). Bonne résolution de dépendances.

Ces outils sont compétents mais n'ont pas atteint la masse critique de `poetry` ou `uv`.

### `uv` (2024–) — la convergence

`uv` est arrivé en 2024 et a rapidement conquis la communauté :
- **Ultra-rapide** : écrit en Rust, 10 à 100x plus rapide que pip
- **Tout-en-un** : remplace pip, venv, pyenv, pipx, pip-tools…
- **Compatible** : lit les `requirements.txt`, `pyproject.toml`, et suit les standards PEP
- **Simple** : moins de choix à faire = moins de confusion

### Les PEP clés du packaging Python

| PEP | Année | Ce qu'elle a introduit |
|-----|-------|----------------------|
| **PEP 241** | 2001 | Métadonnées des paquets (nom, version, auteur…) |
| **PEP 314** | 2003 | Métadonnées v1.1 (ajout de `requires`, `provides`) |
| **PEP 405** | 2012 | `venv` intégré dans Python |
| **PEP 427** | 2012 | Format **wheel** (`.whl`) — remplace le `.egg` |
| **PEP 440** | 2014 | Spécification des numéros de version (`>=1.0,<2.0`) |
| **PEP 453** | 2013 | `pip` inclus avec Python via `ensurepip` |
| **PEP 517** | 2017 | API de build indépendante de `setuptools` |
| **PEP 518** | 2017 | `pyproject.toml` pour déclarer les dépendances de build |
| **PEP 621** | 2020 | Métadonnées du projet dans `pyproject.toml` (`[project]`) |
| **PEP 660** | 2021 | Installation éditable (`pip install -e .`) via `pyproject.toml` |
| **PEP 668** | 2023 | Protection de l'environnement système (pas de `pip install` global sans venv) |
| **PEP 723** | 2024 | Métadonnées inline dans les scripts Python (dépendances dans le script lui-même) |

### Résumé visuel

```
2004  easy_install ──────┐
2008  pip ───────────────┤
2012  conda ─────────────┤    Chacun résolvait un problème,
2017  pipenv ────────────┤    mais ajoutait de la complexité
2018  poetry ────────────┤
2020  hatch, PDM ────────┘
                         │
2024  uv ────────────────── Un seul outil pour (presque) tout
```

</details>